In [ ]:
import torch
from torchvision.models import resnet18
import time

# [변경점] 이제 sys.path.append 없이 바로 import 가능
from module.hardware import AGVHardware
from module.driving_logic import LineTrackingBrain
from module.mission_manager import MissionManager

# 1. 모델 로드
model = resnet18(pretrained=False)
model.fc = torch.nn.Linear(512, 2)
device = torch.device('cuda')
model = model.to(device)
model.load_state_dict(torch.load('best_steering_model_xy_test.pth'))
model = model.eval().half()

# 2. 초기화
agv_hardware = AGVHardware()
agv_brain = LineTrackingBrain(model, device)
manager = MissionManager(agv_hardware, agv_brain)

# 타겟 설정
manager.context.target_plate = "187고1604"

# 3. 메인 루프 실행
try:
    print("🏁 미션 시작!")
    manager.set_state("TRACKING")
    
    while True:
        manager.update()
        time.sleep(0.001)

except KeyboardInterrupt:
    agv_hardware.stop()
    print("정지됨")